In [2]:
import pandas as pd
from dateutil import parser
import pytz
import difflib
from rapidfuzz import fuzz

In [2]:
cfbd = pd.read_csv("/Users/will/GitHub/IS477/data/raw/cfbd_merged.csv", low_memory = False)
box = pd.read_csv("/Users/will/GitHub/IS477/data/raw/cfb_box-scores_2002-2024.csv", low_memory = False)

In [3]:
cfbd.columns

Index(['id_x', 'season', 'week', 'seasonType', 'startDate', 'startTimeTBD',
       'completed', 'neutralSite', 'conferenceGame', 'attendance', 'venueId',
       'venue', 'homeId', 'homeTeam', 'homeClassification', 'homeConference',
       'homePoints', 'homeLineScores', 'homePostgameWinProbability',
       'homePregameElo', 'homePostgameElo', 'awayId', 'awayTeam',
       'awayClassification', 'awayConference', 'awayPoints', 'awayLineScores',
       'awayPostgameWinProbability', 'awayPregameElo', 'awayPostgameElo',
       'excitementIndex', 'highlights', 'notes', 'id_y', 'name', 'capacity',
       'grass', 'dome', 'city', 'state', 'zip', 'countryCode', 'timezone',
       'latitude', 'longitude', 'elevation', 'constructionYear'],
      dtype='object')

In [4]:
print(len(cfbd))
print(len(box))

42294
18909


In [21]:
cfbd.head(10)

,id_x,season,week,seasonType,startDate,startTimeTBD,completed,neutralSite,conferenceGame,attendance,...,dome,city,state,zip,countryCode,timezone,latitude,longitude,elevation,constructionYear
0,222340258,2002,1,regular,2002-08-22T23:30:00.000Z,False,True,False,True,57120.0,...,False,Charlottesville,VA,22904,US,America/New_York,38.031180,-78.513790,170.467743,1931.0
1,222350275,2002,1,regular,2002-08-24T00:00:00.000Z,False,True,False,False,75136.0,...,False,Madison,WI,53711,US,America/Chicago,43.069940,-89.412694,263.604126,1917.0
2,222360194,2002,1,regular,2002-08-24T18:30:00.000Z,False,True,False,False,100037.0,...,False,Columbus,OH,43210,US,America/New_York,40.001645,-83.019727,216.677032,1922.0
3,222360152,2002,1,regular,2002-08-24T20:30:00.000Z,False,True,False,True,47018.0,...,False,Raleigh,NC,27695,US,America/New_York,35.800800,-78.719566,117.008995,1966.0
4,222360158,2002,1,regular,2002-08-24T23:45:00.000Z,False,True,False,False,77779.0,...,False,Lincoln,NE,68588,US,America/Chicago,40.820682,-96.705594,354.931885,1923.0
5,222360066,2002,1,regular,2002-08-25T00:30:00.000Z,False,True,False,False,55132.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,222370259,2002,1,regular,2002-08-25T18:30:00.000Z,False,True,False,False,54016.0,...,False,Blacksburg,VA,24061,US,America/New_York,37.219987,-80.418064,632.263123,1965.0
7,222412309,2002,2,regular,2002-08-29T23:00:00.000Z,False,True,False,False,16073.0,...,False,Kent,OH,44242,US,America/New_York,41.139094,-81.313460,321.364166,1969.0
8,222410189,2002,2,regular,2002-08-29T23:00:00.000Z,False,True,False,False,15696.0,...,False,Bowling Green,OH,43403,US,America/New_York,41.378011,-83.622500,205.905746,1966.0
9,222410058,2002,2,regular,2002-08-29T23:00:00.000Z,False,True,False,False,22074.0,...,False,Tampa,FL,33620,US,America/New_York,27.975869,-82.503334,11.079729,1998.0


In [22]:
box.head(10)

,season,week,date,time_et,game_type,away,home,rank_away,rank_home,conf_away,...,int_away,int_home,pen_num_away,pen_yards_away,pen_num_home,pen_yards_home,possession_away,possession_home,attendance,tv
0,2002,1.0,2002-08-22,7:30 PM,regular,Colorado State,Virginia,NaN,NaN,mwc,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,57120.0,NaN
1,2002,1.0,2002-08-23,8:00 PM,regular,Fresno State,Wisconsin,NaN,25.0,wac,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,75136.0,NaN
2,2002,1.0,2002-08-24,2:30 PM,regular,Texas Tech,Ohio State,NaN,13.0,big12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,100037.0,NaN
3,2002,1.0,2002-08-24,4:30 PM,regular,New Mexico,NC State,NaN,NaN,mwc,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,47018.0,NaN
4,2002,1.0,2002-08-24,7:45 PM,regular,Arizona State,Nebraska,NaN,10.0,pac12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,77779.0,NaN
5,2002,1.0,2002-08-24,8:30 PM,regular,Florida State,Iowa State,3.0,NaN,acc,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,55132.0,NaN
6,2002,1.0,2002-08-25,2:30 PM,regular,Arkansas State,Virginia Tech,NaN,16.0,sun-belt,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,54016.0,NaN
7,2002,2.0,2002-08-29,7:00 PM,regular,Cal Poly SLO,Toledo,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,23074.0,NaN
8,2002,2.0,2002-08-29,7:00 PM,regular,New Hampshire,Kent State,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,16073.0,NaN
9,2002,2.0,2002-08-29,7:00 PM,regular,Richmond,Temple,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15329.0,NaN


In [5]:
box.columns

Index(['season', 'week', 'date', 'time_et', 'game_type', 'away', 'home',
       'rank_away', 'rank_home', 'conf_away', 'conf_home', 'neutral',
       'score_away', 'score_home', 'q1_away', 'q2_away', 'q3_away', 'q4_away',
       'ot_away', 'q1_home', 'q2_home', 'q3_home', 'q4_home', 'ot_home',
       'first_downs_away', 'first_downs_home', 'third_down_comp_away',
       'third_down_att_away', 'third_down_comp_home', 'third_down_att_home',
       'fourth_down_comp_away', 'fourth_down_att_away',
       'fourth_down_comp_home', 'fourth_down_att_home', 'pass_comp_away',
       'pass_att_away', 'pass_yards_away', 'pass_comp_home', 'pass_att_home',
       'pass_yards_home', 'rush_att_away', 'rush_yards_away', 'rush_att_home',
       'rush_yards_home', 'total_yards_away', 'total_yards_home', 'fum_away',
       'fum_home', 'int_away', 'int_home', 'pen_num_away', 'pen_yards_away',
       'pen_num_home', 'pen_yards_home', 'possession_away', 'possession_home',
       'attendance', 'tv'],
      dt

In [10]:
# Full code for data merge, this will be in a .py script later
# Prep functions

def prepare_sched(sched: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare the schedule dataframe (this is your `box` df).

    Expected columns:
      - season, week
      - date (e.g. '2002-08-29')
      - time_et (e.g. '7:00 PM')
      - home, away
      - attendance (optional)
    """
    sched = sched.copy()
    sched = (
        sched.reset_index(drop=True)
             .reset_index()
             .rename(columns={'index': 'sched_game_id'})
    )

    # Local ET datetime from date + time_et
    sched['game_dt_et'] = pd.to_datetime(
        sched['date'].astype(str) + ' ' + sched['time_et'].astype(str),
        errors='coerce'
    )

    sched['date_key'] = sched['game_dt_et'].dt.date
    sched['time_key'] = sched['game_dt_et'].dt.round('5min')

    # Make sure season/week are numeric, but allow NaNs (no int casting!)
    sched['season'] = pd.to_numeric(sched['season'], errors='coerce')
    sched['week'] = pd.to_numeric(sched['week'], errors='coerce')

    # Attendance numeric, allow NaNs, keep as float
    if 'attendance' in sched.columns:
        sched['attendance'] = pd.to_numeric(
            sched['attendance'], errors='coerce'
        ).round()
    else:
        sched['attendance'] = pd.NA

    return sched


def prepare_cfbd_box(cfbd: pd.DataFrame) -> pd.DataFrame:
    """
    Prepare the cfbd games/box dataframe (this is your `cfbd` df).

    Expected columns:
      - season, week
      - startDate (UTC ISO string, e.g. '2002-08-29T23:00:00.000Z')
      - homeTeam, awayTeam
      - attendance (optional)
    """
    cfbd = cfbd.copy()
    cfbd = (
        cfbd.reset_index(drop=True)
            .reset_index()
            .rename(columns={'index': 'cfbd_game_id'})
    )

    # startDate is UTC timestamp
    cfbd['game_dt_utc'] = pd.to_datetime(
        cfbd['startDate'], utc=True, errors='coerce'
    )
    cfbd['game_dt_et'] = (
        cfbd['game_dt_utc']
            .dt.tz_convert('US/Eastern')
            .dt.tz_localize(None)
    )

    cfbd['date_key'] = cfbd['game_dt_et'].dt.date
    cfbd['time_key'] = cfbd['game_dt_et'].dt.round('5min')

    cfbd['season'] = pd.to_numeric(cfbd['season'], errors='coerce')
    cfbd['week'] = pd.to_numeric(cfbd['week'], errors='coerce')

    if 'attendance' in cfbd.columns:
        cfbd['attendance'] = pd.to_numeric(
            cfbd['attendance'], errors='coerce'
        ).round()
    else:
        cfbd['attendance'] = pd.NA

    return cfbd


# Matching helpers

def add_team_match_score(candidates: pd.DataFrame) -> pd.DataFrame:
    """
    Add a fuzzy match score based on home/away team names.

    Requires columns:
      - home_sched, away_sched
      - home_cfbd, away_cfbd
    """
    def _calc(row):
        home_sched = str(row['home_sched'])
        away_sched = str(row['away_sched'])
        home_cfbd = str(row['home_cfbd'])
        away_cfbd = str(row['away_cfbd'])

        h = fuzz.token_sort_ratio(home_sched, home_cfbd)
        a = fuzz.token_sort_ratio(away_sched, away_cfbd)
        row['match_score'] = (h + a) / 2.0
        return row

    return candidates.apply(_calc, axis=1)




def greedy_match(
    candidates: pd.DataFrame,
    score_threshold: float,
    used_sched=None,
    used_cfbd=None
):
    """
    Greedy 1–1 matching:
      - sort by match_score desc
      - keep best available pairs
      - each sched_game_id and cfbd_game_id appears at most once
    """
    if used_sched is None:
        used_sched = set()
    if used_cfbd is None:
        used_cfbd = set()

    if candidates.empty:
        return pd.DataFrame(), used_sched, used_cfbd

    candidates = candidates.sort_values('match_score', ascending=False).copy()
    rows = []

    for r in candidates.itertuples(index=False):
        if r.match_score < score_threshold:
            continue
        if r.sched_game_id in used_sched or r.cfbd_game_id in used_cfbd:
            continue

        used_sched.add(r.sched_game_id)
        used_cfbd.add(r.cfbd_game_id)
        rows.append(r._asdict())

    if rows:
        matches_df = pd.DataFrame(rows)
    else:
        matches_df = pd.DataFrame(columns=candidates.columns)

    return matches_df, used_sched, used_cfbd


# Main function: match + merge

def match_cfbd_box(
    cfbd_raw: pd.DataFrame,
    box_raw: pd.DataFrame,
    time_window_att: float = 15,  # minutes for attendance-based candidates
    score_thr_att: float = 80,    # fuzzy threshold for attendance-based
    time_window_time: float = 10, # minutes for time-based candidates
    score_thr_time: float = 85    # fuzzy threshold for time-based
) -> pd.DataFrame:
    """
    Match cfbd and box dataframes and return a merged dataframe.

    ARGUMENTS:
      - cfbd_raw: dataframe with startDate, homeTeam, awayTeam, attendance,
      - box_raw:  dataframe with date, time_et, home, away, attendance,

    - Uses season, week, date, time, attendance, and fuzzy team names.
    - Team names in the final result default to the schedule (box_raw) home/away.
    - Only games that match are included.
    """

    # Prep
    sched = prepare_sched(box_raw)       # schedule: date + time_et + home/away
    cfbd = prepare_cfbd_box(cfbd_raw)    # cfbd: startDate + homeTeam/awayTeam

    # Minimal views for matching
    sched_small = sched[['sched_game_id', 'season', 'week', 'date_key', 'time_key',
                         'home', 'away', 'attendance']]
    cfbd_small = cfbd[['cfbd_game_id', 'season', 'week', 'date_key', 'time_key',
                       'homeTeam', 'awayTeam', 'attendance']]


    # 1) Attendance-based candidates

    sched_att = sched_small.dropna(subset=['attendance'])
    cfbd_att = cfbd_small.dropna(subset=['attendance'])

    cand_att = sched_att.merge(
        cfbd_att,
        on=['season', 'week', 'date_key', 'attendance'],
        suffixes=('_sched', '_cfbd')
    )

    if not cand_att.empty:
        cand_att = cand_att.rename(columns={
            'home': 'home_sched',
            'away': 'away_sched',
            'homeTeam': 'home_cfbd',
            'awayTeam': 'away_cfbd'
        })

        # Attach exact datetimes for time window filter
        cand_att = cand_att.merge(
            sched[['sched_game_id', 'game_dt_et']],
            on='sched_game_id'
        ).merge(
            cfbd[['cfbd_game_id', 'game_dt_et']],
            on='cfbd_game_id',
            suffixes=('_sched_dt', '_cfbd_dt')
        )

        cand_att['time_diff_min'] = (
            (cand_att['game_dt_et_sched_dt'] - cand_att['game_dt_et_cfbd_dt'])
            .abs()
            .dt.total_seconds() / 60.0
        )

        cand_att = cand_att[cand_att['time_diff_min'] <= time_window_att]

        if not cand_att.empty:
            cand_att = add_team_match_score(cand_att)
            matches_att, used_sched, used_cfbd = greedy_match(
                cand_att, score_thr_att
            )
        else:
            matches_att = pd.DataFrame()
            used_sched, used_cfbd = set(), set()
    else:
        matches_att = pd.DataFrame()
        used_sched, used_cfbd = set(), set()

    # 2) Time-based candidates (no attendance)
    sched_small_rem = sched_small[~sched_small['sched_game_id'].isin(used_sched)]
    cfbd_small_rem = cfbd_small[~cfbd_small['cfbd_game_id'].isin(used_cfbd)]

    cand_time = sched_small_rem.merge(
        cfbd_small_rem,
        on=['season', 'week', 'date_key', 'time_key'],
        suffixes=('_sched', '_cfbd')
    )

    if not cand_time.empty:
        cand_time = cand_time.rename(columns={
            'home': 'home_sched',
            'away': 'away_sched',
            'homeTeam': 'home_cfbd',
            'awayTeam': 'away_cfbd'
        })

        cand_time = cand_time.merge(
            sched[['sched_game_id', 'game_dt_et']],
            on='sched_game_id'
        ).merge(
            cfbd[['cfbd_game_id', 'game_dt_et']],
            on='cfbd_game_id',
            suffixes=('_sched_dt', '_cfbd_dt')
        )

        cand_time['time_diff_min'] = (
            (cand_time['game_dt_et_sched_dt'] - cand_time['game_dt_et_cfbd_dt'])
            .abs()
            .dt.total_seconds() / 60.0
        )

        cand_time = cand_time[cand_time['time_diff_min'] <= time_window_time]

        if not cand_time.empty:
            cand_time = add_team_match_score(cand_time)
            matches_time, used_sched, used_cfbd = greedy_match(
                cand_time, score_thr_time, used_sched, used_cfbd
            )
        else:
            matches_time = pd.DataFrame()
    else:
        matches_time = pd.DataFrame()

    # 3) Combine matches and build final merged dataframe
    matches = pd.concat([matches_att, matches_time], ignore_index=True)

    if matches.empty:
        # No matches found
        return pd.DataFrame()

    sched_full = sched.set_index('sched_game_id')
    cfbd_full = cfbd.set_index('cfbd_game_id')

    merged = matches.join(
        sched_full, on='sched_game_id', rsuffix='_sched_full'
    ).join(
        cfbd_full, on='cfbd_game_id', rsuffix='_cfbd_full'
    )

    # Drop cfbd team name columns if you want schedule names as canonical
    merged = merged.drop(columns=['homeTeam', 'awayTeam'], errors='ignore')
    # merged = merged.rename(columns={'home': 'homeTeam', 'away': 'awayTeam'})

    return merged

In [16]:
merged_games = match_cfbd_box(cfbd, box)

/var/folders/8l/f7sy6_5s2f76flj4cpkqxftr0000gn/T/ipykernel_4167/3909872013.py:24: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  sched['game_dt_et'] = pd.to_datetime(


   season  week    date_key        home            away
0    2002   1.0  2002-08-22    Virginia  Colorado State
1    2018   7.0  2018-10-13      Auburn       Tennessee
2    2018   6.0  2018-10-06  Ohio State         Indiana
3    2018   6.0  2018-10-06        UNLV      New Mexico
4    2018   6.0  2018-10-06    Colorado   Arizona State


In [23]:
print(merged_games[['season', 'week', 'date_key', 'home', 'away']].sort_values('season').head(100))

      season  week    date_key            home            away
0       2002   1.0  2002-08-22        Virginia  Colorado State
8574    2002   3.0  2002-09-07      Miami (OH)            Iowa
8573    2002   3.0  2002-09-07      Pittsburgh       Texas A&M
8572    2002   3.0  2002-09-07   Southern Miss        Illinois
8571    2002   3.0  2002-09-05          Temple    Oregon State
...      ...   ...         ...             ...             ...
8480    2002   5.0  2002-09-21       Texas A&M   Virginia Tech
8479    2002   5.0  2002-09-21          Oregon  Portland State
8478    2002   5.0  2002-09-21  Michigan State      Notre Dame
8477    2002   5.0  2002-09-21    Georgia Tech             BYU
8476    2002   5.0  2002-09-21            UCLA        Colorado

[100 rows x 5 columns]


In [17]:
len(merged_games)

16547

In [14]:
merged_games.to_csv("/Users/will/GitHub/IS477/data/cleaned/merged_games.csv")

In [4]:
df = pd.read_csv("/Users/will/GitHub/IS477/data/cleaned/merged_games.csv")

/var/folders/8l/f7sy6_5s2f76flj4cpkqxftr0000gn/T/ipykernel_30521/2849834276.py:1: DtypeWarning: Columns (5,10,17,116) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("/Users/will/GitHub/IS477/data/cleaned/merged_games.csv")


In [6]:
pd.set_option('display.max_seq_items', None)  # don't truncate index/columns display
print(df.columns)

Index(['Unnamed: 0', 'sched_game_id', 'season', 'week', 'date_key',
       'time_key_sched', 'home_sched', 'away_sched', 'attendance',
       'cfbd_game_id', 'time_key_cfbd', 'home_cfbd', 'away_cfbd',
       'game_dt_et_sched_dt', 'game_dt_et_cfbd_dt', 'time_diff_min',
       'match_score', 'time_key', 'attendance_sched', 'attendance_cfbd',
       'season_sched_full', 'week_sched_full', 'date', 'time_et', 'game_type',
       'away', 'home', 'rank_away', 'rank_home', 'conf_away', 'conf_home',
       'neutral', 'score_away', 'score_home', 'q1_away', 'q2_away', 'q3_away',
       'q4_away', 'ot_away', 'q1_home', 'q2_home', 'q3_home', 'q4_home',
       'ot_home', 'first_downs_away', 'first_downs_home',
       'third_down_comp_away', 'third_down_att_away', 'third_down_comp_home',
       'third_down_att_home', 'fourth_down_comp_away', 'fourth_down_att_away',
       'fourth_down_comp_home', 'fourth_down_att_home', 'pass_comp_away',
       'pass_att_away', 'pass_yards_away', 'pass_comp_home', '